# Détection de contours sur Stanford Dogs

Ce notebook applique différents algorithmes de détection de contours sur des images du dataset :
- **scikit-image** : Sobel, Canny, Prewitt, Roberts, Laplace
- **Kornia** (différentiable, GPU) : Sobel, Canny, Laplacian, filtres gradient
- **OpenCV** : Canny, Sobel, Laplacian

Comparaison visuelle côte à côte.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/SPeillet/bcs_determination")
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from PIL import Image
from glob import glob
import random

import torch
import kornia
from kornia import feature as kfeature
from kornia import filters as kfilters

import skimage
from skimage import filters as skfilters
from skimage import feature as skfeature
from skimage import color as skcolor

import cv2

plt.rcParams["font.size"] = 10
sns_style = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"scikit-image: {skimage.__version__}")
print(f"kornia:       {kornia.__version__}")
print(f"OpenCV:       {cv2.__version__}")
print(f"Device:       {device}")

## 1. Chargement d'images samples

In [ ]:
DATA_DIR = ROOT / "data" / "stanford_dogs" / "Images"
SEED = 42
random.seed(SEED)

all_images = sorted(glob(str(DATA_DIR / "*" / "*.jpg")))
sample_paths = random.sample(all_images, 4)

def load_rgb(path, size=400):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size), Image.LANCZOS)
    return np.array(img)

def get_breed_name(path):
    folder = Path(path).parent.name
    return "-".join(folder.split("-")[1:])

samples = []
for p in sample_paths:
    img = load_rgb(p)
    gray = skcolor.rgb2gray(img)
    samples.append({"path": p, "rgb": img, "gray": gray, "breed": get_breed_name(p)})

fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
for ax, s in zip(axes, samples):
    ax.imshow(s["rgb"])
    ax.set_title(s["breed"], fontsize=10)
    ax.axis("off")
fig.suptitle("Original Samples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Détection de contours — scikit-image

In [ ]:
methods_sk = {
    "Sobel":       lambda g: skfilters.sobel(g),
    "Prewitt":     lambda g: skfilters.prewitt(g),
    "Roberts":     lambda g: skfilters.roberts(g),
    "Scharr":      lambda g: skfilters.scharr(g),
    "Farid":       lambda g: skfilters.farid(g),
    "Laplace":     lambda g: skfilters.laplace(g),
    "Canny (σ=1)": lambda g: skfeature.canny(g, sigma=1.0),
    "Canny (σ=2)": lambda g: skfeature.canny(g, sigma=2.0),
}

for s in samples:
    gray = s["gray"]
    n_methods = len(methods_sk)
    cols = 4
    rows = (n_methods + 1 + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flatten()

    axes_flat[0].imshow(gray, cmap="gray")
    axes_flat[0].set_title("Original (gray)")
    axes_flat[0].axis("off")

    for i, (name, fn) in enumerate(methods_sk.items(), start=1):
        edges = fn(gray)
        axes_flat[i].imshow(edges, cmap="gray")
        axes_flat[i].set_title(name)
        axes_flat[i].axis("off")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(f"scikit-image — {s['breed']}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 3. Détection de contours — Kornia (GPU, différentiable)

In [ ]:
from kornia.filters import canny as kornia_canny

def to_kornia_batch(rgb_np):
    img = torch.from_numpy(rgb_np).permute(2, 0, 1).float() / 255.0
    return img.unsqueeze(0).to(device)

def to_gray_torch(rgb_batch):
    return kornia.color.rgb_to_grayscale(rgb_batch)

for s in samples:
    batch = to_kornia_batch(s["rgb"])
    gray_batch = to_gray_torch(batch)

    sobel_edges = kfilters.sobel(gray_batch)
    spatial_grad = kfilters.spatial_gradient(gray_batch, order=1)
    grad_mag = torch.sqrt(spatial_grad[:, :, 0, :, :] ** 2 + spatial_grad[:, :, 1, :, :] ** 2)
    laplacian_edges = kfilters.laplacian(gray_batch, kernel_size=3)
    canny_edges, _ = kornia_canny(gray_batch, low_threshold=0.1, high_threshold=0.3)
    gaussian_blur = kfilters.gaussian_blur2d(gray_batch, (5, 5), (1.5, 1.5))
    unsharp = gray_batch - gaussian_blur

    kornia_results = {
        "Sobel":           sobel_edges[0, 0],
        "Gradient magnitude": grad_mag[0, 0],
        "Laplacian":       laplacian_edges[0, 0],
        "Canny":           canny_edges[0, 0],
        "Gaussian blur":   gaussian_blur[0, 0],
        "Unsharp mask":    unsharp[0, 0],
    }

    n = len(kornia_results)
    cols = 4
    rows = (n + 1 + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flatten()

    axes_flat[0].imshow(gray_batch[0, 0].cpu().numpy(), cmap="gray")
    axes_flat[0].set_title("Original (gray)")
    axes_flat[0].axis("off")

    for i, (name, tensor) in enumerate(kornia_results.items(), start=1):
        axes_flat[i].imshow(tensor.cpu().numpy(), cmap="gray")
        axes_flat[i].set_title(name)
        axes_flat[i].axis("off")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(f"Kornia (GPU) — {s['breed']}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 4. Détection de contours — OpenCV

In [ ]:
for s in samples:
    gray = (s["gray"] * 255).astype(np.uint8)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)

    laplacian = cv2.Laplacian(gray, cv2.CV_64F)

    canny_low = cv2.Canny(gray, 50, 100)
    canny_mid = cv2.Canny(gray, 100, 200)
    canny_high = cv2.Canny(gray, 150, 300)

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    canny_blur = cv2.Canny(blurred, 100, 200)

    cv_results = {
        "Sobel magnitude": sobel_mag,
        "Laplacian":       laplacian,
        "Canny (50-100)":  canny_low,
        "Canny (100-200)": canny_mid,
        "Canny (150-300)": canny_high,
        "Canny+Blur":      canny_blur,
    }

    n = len(cv_results)
    cols = 4
    rows = (n + 1 + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flatten()

    axes_flat[0].imshow(gray, cmap="gray")
    axes_flat[0].set_title("Original (gray)")
    axes_flat[0].axis("off")

    for i, (name, img) in enumerate(cv_results.items(), start=1):
        axes_flat[i].imshow(img, cmap="gray")
        axes_flat[i].set_title(name)
        axes_flat[i].axis("off")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(f"OpenCV — {s['breed']}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5. Comparaison directe : Canny (skimage vs Kornia vs OpenCV)

In [ ]:
fig, axes = plt.subplots(len(samples), 4, figsize=(20, 5 * len(samples)))

for row, s in enumerate(samples):
    gray_f = s["gray"]
    gray_u8 = (gray_f * 255).astype(np.uint8)

    canny_sk = skfeature.canny(gray_f, sigma=2.0)

    batch = to_kornia_batch(s["rgb"])
    gray_t = to_gray_torch(batch)
    canny_k, _ = kornia_canny(gray_t, low_threshold=0.08, high_threshold=0.25)
    canny_k_np = canny_k[0, 0].cpu().numpy()

    canny_cv = cv2.Canny(gray_u8, 100, 200)

    axes[row, 0].imshow(s["rgb"])
    axes[row, 0].set_title(f"{s['breed']}")
    axes[row, 1].imshow(canny_sk, cmap="gray")
    axes[row, 1].set_title("Canny (skimage, σ=2)")
    axes[row, 2].imshow(canny_k_np, cmap="gray")
    axes[row, 2].set_title("Canny (Kornia, GPU)")
    axes[row, 3].imshow(canny_cv, cmap="gray")
    axes[row, 3].set_title("Canny (OpenCV, 100-200)")

for ax in axes.flatten():
    ax.axis("off")

fig.suptitle("Comparaison Canny : scikit-image vs Kornia vs OpenCV", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Overlay : contours superposés à l'image originale

In [ ]:
def overlay_edges(rgb, edges_gray, color=(0, 255, 0), alpha=0.6):
    overlay = rgb.copy()
    mask = edges_gray > 0
    if mask.dtype != bool:
        mask = mask > (edges_gray.max() * 0.3) if edges_gray.max() > 0 else mask > 0
    overlay[mask] = (1 - alpha) * rgb[mask] + alpha * np.array(color)
    return overlay.astype(np.uint8)

fig, axes = plt.subplots(len(samples), 3, figsize=(15, 5 * len(samples)))

for row, s in enumerate(samples):
    gray_f = s["gray"]
    gray_u8 = (gray_f * 255).astype(np.uint8)
    rgb = s["rgb"].copy()

    canny_sk = skfeature.canny(gray_f, sigma=2.0).astype(np.float64) * 255
    canny_cv = cv2.Canny(gray_u8, 100, 200).astype(np.float64)

    batch = to_kornia_batch(s["rgb"])
    gray_t = to_gray_torch(batch)
    sobel_k = kfilters.sobel(gray_t)[0, 0].cpu().numpy()
    sobel_k = (sobel_k / sobel_k.max() * 255) if sobel_k.max() > 0 else sobel_k

    axes[row, 0].imshow(overlay_edges(rgb, canny_sk, color=(0, 255, 0)))
    axes[row, 0].set_title("Canny (skimage) overlay")
    axes[row, 1].imshow(overlay_edges(rgb, canny_cv, color=(255, 100, 0)))
    axes[row, 1].set_title("Canny (OpenCV) overlay")
    axes[row, 2].imshow(overlay_edges(rgb, sobel_k, color=(0, 150, 255)))
    axes[row, 2].set_title("Sobel (Kornia) overlay")

for ax in axes.flatten():
    ax.axis("off")

fig.suptitle("Edge Overlay on Original Images", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. Multi-scale Canny (scikit-image)

In [ ]:
sigmas = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]

for s in samples[:2]:
    fig, axes = plt.subplots(1, len(sigmas) + 1, figsize=(4 * (len(sigmas) + 1), 4))
    axes[0].imshow(s["gray"], cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    for i, sigma in enumerate(sigmas, 1):
        edges = skfeature.canny(s["gray"], sigma=sigma)
        axes[i].imshow(edges, cmap="gray")
        axes[i].set_title(f"Canny σ={sigma}")
        axes[i].axis("off")

    fig.suptitle(f"Multi-scale Canny — {s['breed']}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()